# STAC AI Workshop · NYC · 21 October 2026

## 06 · Framing stability versus evidence reliability

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Inspect exact saved model responses rather than illustrative answers
2. Compare matched framings with identical evidence
3. Recompute evidence-rule violations and aggregate correctness
4. Distinguish a policy dependency from causal model attribution

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/06_evaluation.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/06_evaluation.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:4** | `notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

## Class Exercise · a saved, measured experiment
This matches **Web Section 6**. The file contains 1,200 completed Together.ai assessments: 100 synthetic cases × 3 framings × 2 models × 2 repeats. All cells replay saved outputs; no API calls are made.

The larger study **did not reproduce** the pilot's increased unsupported-buy rate under bullish framing. We inspect what actually happened, including the negative result.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `data=load_fixture("framing-explorer.json")` | Load a saved teaching or recorded-result fixture; this is not a live model call. |
| **C02:6** | `check=check_framing_evidence(case)` | Apply the explicit evidence requirements; distinguish this rule check from model generation. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
data=load_fixture("framing-explorer.json")
model=data["manifest"]["models"][0]
case_id="alpha-060"  # same default conflicting-evidence case as the website
case=next(c for c in data["cases"] if c["id"]==case_id)
check=check_framing_evidence(case)
show_table([{"Required action":check["action"],"Failed dependencies":", ".join(check["failed"]),"Actual values":str(check["values"]["actual_eps"]),"Consensus values":str(check["values"]["consensus_eps"])}])
show_table(json.loads(case["evidence"]))
print("Decision rule:",case["rules"])

## Compare exact responses
Change `model` to the second model or `case_id` to another case and rerun. Declared source IDs are model citations; they do not prove causal dependence on those records.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:3** | `for frame in ("neutral","bullish","skeptical"):` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
from IPython.display import display, Markdown
for frame in ("neutral","bullish","skeptical"):
    r=next(r for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==frame and r["repeat"]==1)
    display(Markdown("### "+frame+" · saved action: "+r["action"]+"\n\n```text\n"+r["output"]+"\n```"))
    print("Matches rule:",r["action"]==check["action"],"| cited source IDs:",", ".join(r["source_ids"]) or "none")
show_table([{"Framing":f,"Repeat 1":next(r["action"] for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==f and r["repeat"]==1),"Repeat 2":next(r["action"] for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==f and r["repeat"]==2)} for f in ("neutral","bullish","skeptical")])

## All-case results
Two repeats per case are not 200 independent scenarios. Always inspect aggregate results before generalizing from a selected failure.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:5** | `metrics=framing_metrics(data,model_name,frame)` | Recompute a metric from the saved evaluation cases, not a new training run. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
rows=[]
for model_name in data["manifest"]["models"]:
    for frame in ("neutral","bullish","skeptical"):
        metrics=framing_metrics(data,model_name,frame)
        rows.append({"Model":model_name,"Framing":frame,"Correct":f'{metrics["correct"]}/{metrics["n"]}',"Unsupported BUY":metrics["unsupported_buys"]})
show_table(rows)
assert len(data["outputs"])==1200
print("Qwen evidence failures persisted across framing; Llama passed these defined cases.")

## Controlled evidence removal · rule check only
Remove actual-EPS records, then recompute the required action. The saved model response still belongs to the original packet. This does not rerun the model and is not causal attribution of its behaviour.

<!-- WORKSHOP-CODE-GUIDE C05 -->
### Code cell C05 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C05:3** | `removed=check_framing_evidence(case,without_actual)` | Apply the explicit evidence requirements; distinguish this rule check from model generation. |
| **C05:6** | `references=load_fixture("section567-reference.json")["framing"]` | Load a saved teaching or recorded-result fixture; this is not a live model call. |
| **C05:10** | `for ref in references["metrics"]:assert framing_metrics(data,ref["model"],ref["frame"])==ref["result"]` | Recompute a metric from the saved evaluation cases, not a new training run. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C05 · code cell
without_actual=[r for r in json.loads(case["evidence"]) if r["kind"]!="actual_eps"]
removed=check_framing_evidence(case,without_actual)
show_table([{"Condition":"Original packet","Required action":check["action"],"Failed":", ".join(check["failed"])},{"Condition":"Actual EPS removed","Required action":removed["action"],"Failed":", ".join(removed["failed"])}])
assert "actual_eps:missing" in removed["failed"]
references=load_fixture("section567-reference.json")["framing"]
for ref in references["checks"]:
    c=next(c for c in data["cases"] if c["id"]==ref["id"])
    assert check_framing_evidence(c)==ref["result"]
for ref in references["metrics"]:assert framing_metrics(data,ref["model"],ref["frame"])==ref["result"]
import hashlib
for c in data["cases"]:
    assert hashlib.sha256(c["evidence"].encode()).hexdigest()==c["evidence_sha256"]
    assert all(r["evidence_sha256"]==c["evidence_sha256"] for r in data["outputs"] if r["case"]==c["id"])
print("PASS: 100 evidence checks, 6 aggregate rows and 1,200 evidence identities verified.")

## Learning check
Can an always-wrong model be framing-stable? **Yes.** Score correctness and evidence-rule compliance as well as sensitivity.

Does removing evidence from the checker prove that the LLM used it? **No.** That requires controlled model reruns, not merely changed policy requirements.

These are measured responses to synthetic cases—not general population bias rates or investment alpha.

## Main takeaway
Consistent answers are not independently validated answers. The graph checks a declared evidence requirement; causal attribution and calibration need additional experiments.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.

## Answer checks · default configuration

- The file contains **1,200 saved assessments**; reading it is not a new provider call. Two repeats or several framings of one case are not independent scenarios.
- Compare the saved actions with the independently applied evidence rule, not with the model’s explanation of its own correctness.
- The expanded test did not show bullish framing increasing unsupported buys. Qwen still violated some evidence requirements; Llama passed this defined benchmark. This does not prove universal immunity to framing.
- Removing actual-EPS records must flag **`actual_eps:missing`** in the rule check. The saved response is **not regenerated** by that removal, so this is not evidence that the model changed its answer.
- Consistent answers can be consistently wrong. Selected-action stability does not establish unchanged token probabilities or a neural explanation.
- Assertions verify **100 evidence checks, 6 aggregate rows and 1,200 evidence identities** against the saved reference.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.